# 67. 구조화 조회의 대상 표현과 추출기 비교

67A는 저장된 144개 추출 계획을 고정하고 조회 규칙만 비교한다. 67B는 선택한 DB·매칭 규칙을 고정한 뒤 기존·수정 추출기를 새로 비교한다. 개발 평가이며 운영 코드·원문·기존 DB는 변경하지 않는다.

공통 SQL 연결은 `scripts/67_query_repair.py`로 재사용한다. 노트북이 실험 조건·입출력·선택을 소유하며, helper는 68·69에서도 같은 조회 규칙을 쓰기 위해 둔다. HTML은 결과 검토 후 직접 작성한다.

In [1]:
from pathlib import Path
import copy, importlib.util, json, sqlite3, time
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
spec = importlib.util.spec_from_file_location('query67_notebook', ROOT/'notebooks/scripts/67_query_repair.py')
K = importlib.util.module_from_spec(spec); spec.loader.exec_module(K)
R = K.load_runtime()
from hashlib import sha256
BASE = ROOT/'notebooks/data/67_structured_retrieval_repair'
BASE.mkdir(exist_ok=True)
read = lambda p: json.loads(Path(p).read_text())
def save(name, value):
    path = BASE/name
    if path.exists():
        assert read(path) == value, '기존 결과와 다름: '+name
    else:
        path.write_text(json.dumps(value, ensure_ascii=False, indent=2)+'\n')
    return value
sha = lambda p: sha256(Path(p).read_bytes()).hexdigest()
directory = R.M.DB_NEW
facts = R.R.fact_map(directory)
documents = [json.loads(s) for s in (directory/'documents.jsonl').read_text().splitlines()]
plans = read(R.B65/'routing_diagnostics.json')['records']
previous = read(R.B65/'retrieval_results.json')['records']
old_by_query_repeat = {(x['query_id'], x['repeat']): x for x in previous if x['mode']=='SEQ'}
assert len(plans) == 144
print('저장 계획', len(plans), '카드', len(documents), '혜택행', len(facts), '신규 API 0')

저장 계획 144 카드 106 혜택행 3403 신규 API 0


## 67A: 원자료와 DB 보존·일치 확인

현재 65번 기준 DB는 60번에서 정규화한 `db_v3`다. 여기서는 DB 값을 바꾸지 않고 읽기 전용으로 연다. SQLite에 저장된 원본 JSON과 정규화 필드가 JSONL 내보내기와 같은지 모든 행을 확인한다. 이는 PDF 전체 추출 정확도의 증명과는 다르다.

In [2]:
bound_paths = [directory/name for name in ('structured.sqlite', 'facts.jsonl', 'documents.jsonl')]
bound_paths += [R.B65/name for name in ('routing_diagnostics.json', 'parser_results.json', 'retrieval_results.json')]
bound_paths += [ROOT/'notebooks/scripts'/name for name in ('67_query_repair.py', '67_test_query_repair.py', '60_query_repair.py', '65_scope_repair.py')]
inputs = {str(p.relative_to(ROOT)): sha(p) for p in bound_paths}
save('67A_input_contract.json', {'stage':'saved_plan_replay', 'files_sha256':inputs,
    'matcher_versions':['baseline60_literal', K.VERSION], 'DB_changed':False,
    'notebook_phase_code_sha256':sha256(json.dumps([(c['id'],c['source']) for c in read(ROOT/'notebooks/67_structured_retrieval_repair.ipynb')['cells'] if c['cell_type']=='code' and c['id'].startswith('exp67-A-')],ensure_ascii=False,sort_keys=True).encode()).hexdigest(),
    'new_API_calls':0, 'queries':48, 'repeats':3, 'plans':144,
    'candidate_rule_chosen_after_observing_development_zero_cases':True,
    'caution':'후행 개발 규칙이며 별도 69 확인 전 일반화·운영 채택 근거가 아니다.'})
with sqlite3.connect((directory/'structured.sqlite').resolve().as_uri()+'?mode=ro&immutable=1', uri=True) as database:
    database.execute('PRAGMA query_only=ON')
    rows = database.execute('SELECT fact_id,document_id,raw_json,derived_json FROM facts ORDER BY fact_id').fetchall()
    assert len(rows) == len(facts)
    assert {r[0] for r in rows} == set(facts)
    for fid, card, raw, derived in rows:
        assert facts[fid]['document_id'] == card
        assert json.loads(raw) == facts[fid]['raw_fact']
        assert json.loads(derived) == facts[fid]['derived']
save('67A_DB_export_consistency.json', {'status':'all_facts_equal', 'facts':len(rows),
    'raw_fact_equal':True, 'derived_equal':True, 'card_owner_equal':True,
    'source_files_sha256':{str(p.relative_to(ROOT)):sha(p) for p in bound_paths[:3]},
    'limitation':'추출 JSON의 표·공통조건 연결 정확도 자체는 이 등가 검사로 증명하지 않는다.'})
print('SQLite와 JSONL 동일', len(rows), '행')

SQLite와 JSONL 동일 3403 행


## 고정 계획의 전체 재생

추출 결과를 다시 생성하지 않는다. 같은 저장 계획·DB·발급사/카드 범위에서 기존 규칙과 좁은 대상 후보 보완만 바꾼다. ‘진료비’, ‘회비’ 뒤가 아닌 앞의 명시 대상도 후보로 허용한다. 제외·숫자·같은 카드 AND는 유지한다. 기차→철도, 극장→영화 같은 새 동의어 보정은 하지 않는다.

카드 범위를 줄인 뒤의 비정형 검색은 아직 이 단계의 측정 대상이 아니다. 아래 시간은 로컬 구조화 SQL 조회 시간이다.

In [3]:
replay_path = BASE/'67A_replay_results.json'
if replay_path.exists():
    replay = read(replay_path)
    assert replay['input_contract_sha256'] == sha(BASE/'67A_input_contract.json')
else:
    comparisons=[]; started=time.perf_counter(); cpu=time.process_time()
    for index, record in enumerate(plans):
        qid, repeat = record['query_id'], record['repeat']
        original = R.S.original_scope(record['question'], documents)
        assert original == old_by_query_repeat[qid, repeat]['original_scope']
        outputs={}
        order = ['baseline','projected'] if index%2 else ['projected','baseline']
        for matcher in order:
            outputs[matcher] = K.route(R, record['plan'], original, directory, facts,
                matcher=matcher, parser_failed=record['parser_status']!='ok')
        old_cards = set(old_by_query_repeat[qid, repeat]['routing']['eligible_cards'])
        assert set(outputs['baseline']['eligible_cards']) == old_cards
        new_cards=set(outputs['projected']['eligible_cards'])
        comparisons.append({'query_id':qid,'repeat':repeat,'question':record['question'],
            'original_scope':original,'saved_plan':record['plan'],'outputs':outputs,
            'added_cards':sorted(new_cards-old_cards),'lost_cards':sorted(old_cards-new_cards),
            'previous_positive':bool(old_cards),'changed_cards':old_cards!=new_cards})
        if (index+1)%24==0: print('재생',index+1,'/144')
    replay={'status':'replayed_awaiting_manual_candidate_audit', 'records':comparisons,
        'input_contract_sha256':sha(BASE/'67A_input_contract.json'),
        'wall_seconds':time.perf_counter()-started,'cpu_seconds':time.process_time()-cpu,
        'new_API_calls':0}
    save('67A_replay_results.json', replay)
for name, expected in inputs.items(): assert sha(ROOT/name) == expected
records=replay['records']
print('기존 양성',sum(r['previous_positive'] for r in records),
      '보완 후 양성',sum(bool(r['outputs']['projected']['eligible_cards']) for r in records),
      '변경',sum(r['changed_cards'] for r in records),
      '기존 양성 카드손실',sum(bool(r['lost_cards']) for r in records))

기존 양성 100 보완 후 양성 108 변경 8 기존 양성 카드손실 0


## 수동 정답 계획 진단 및 추출기 후속 비교

기존 0건의 원인을 대상 입도·행위·정규화·관계·실제 부재로 나누고, 양성 계획도 변경된 카드와 조건을 검토한다. 수동 계획은 DB 표현 가능성 진단용이며 자동 검색·답변의 입력으로 쓰지 않는다.

67B의 신규 추출 비교 셀은 66 평가 완료와 67A 후보 검토 후 추가한다. 현재 결과는 조회 규칙의 준비·개발 재생 단계이며 최종 설정 선택이 아니다.

## 67A 보완: 같은 행의 정규 실적 수치 정규화

문자 대상 문제와 별개로, 조건 본문에는 정규 전월실적이 있지만 수치 필드가 `unknown`인 행을 조사한다. 여러 금액 중 거래금액과 전월실적 금액을 구분하지 못하거나 ‘전월 20만원 이상’ 같은 표현이 누락된 경우다.

실험 DB 사본에서만, 같은 행의 명시 문구·타입 금액·조건 출처가 일치하는 단일 `이상` 조건을 보완한다. 범위·신규 유예·연간/복수 기간·엄격 초과·미기재는 보완하지 않는다. 원본 JSON 필드·관계·기존 known/0원은 보존하며 공통조건을 다른 행으로 추측 전파하지 않는다. OCR 및 JSON의 행 연결을 신뢰한다는 기존 실험 전제 아래의 수치 파생값 수정이다.

In [4]:
import resource, shutil
spec = importlib.util.spec_from_file_location('numeric67_notebook', ROOT/'notebooks/scripts/67_db_repair.py')
D = importlib.util.module_from_spec(spec); spec.loader.exec_module(D)
prototype = BASE/'db_v4'
patches = [patch for fact in facts.values() if (patch := D.spend_patch(fact))]
patch_by_id = {p['fact_id']:p for p in patches}
db_phase_sources = {**inputs, **{str((ROOT/'notebooks/scripts'/name).relative_to(ROOT)):sha(ROOT/'notebooks/scripts'/name)
    for name in ('67_db_repair.py','67_test_db_repair.py')}}
db_code = [(c['id'],c['source']) for c in read(ROOT/'notebooks/67_structured_retrieval_repair.ipynb')['cells']
           if c['cell_type']=='code' and c['id'].startswith('exp67-DB-')]
db_contract={'input_files_sha256':db_phase_sources,'normalization_rule':D.VERSION,
    'notebook_phase_code_sha256':sha256(json.dumps(db_code,ensure_ascii=False,sort_keys=True).encode()).hexdigest(),
    'patches':patches,'OCR_and_JSON_row_relations_assumed_valid':True,
    'not_source_table_relationship_validation':True,'new_API_calls':0}
save('67A_DB_repair_contract.json',db_contract)
manifest_path=BASE/'67A_DB_copy_manifest.json'
if manifest_path.exists():
    copy_manifest=read(manifest_path)
    assert copy_manifest['repair_contract_sha256']==sha(BASE/'67A_DB_repair_contract.json')
    for name,expected in copy_manifest['output_files_sha256'].items():assert sha(ROOT/name)==expected
else:
    assert not prototype.exists(),'불완전한 신규 사본을 먼저 확인하세요. 기존 원본은 수정하지 않습니다.'
    started=time.perf_counter(); cpu=time.process_time();prototype.mkdir()
    modified=[]
    for fact in facts.values():
        new=copy.deepcopy(fact)
        if fact['fact_id'] in patch_by_id:new['derived'].update(patch_by_id[fact['fact_id']]['updates'])
        modified.append(new)
    (prototype/'facts.jsonl').write_text(''.join(json.dumps(f,ensure_ascii=False,sort_keys=True)+'\n' for f in modified))
    shutil.copyfile(directory/'documents.jsonl',prototype/'documents.jsonl')
    with sqlite3.connect((directory/'structured.sqlite').resolve().as_uri()+'?mode=ro&immutable=1',uri=True) as old_db, sqlite3.connect(prototype/'structured.sqlite') as new_db:
        old_db.backup(new_db)
        before={row[0]:row for row in old_db.execute('SELECT * FROM facts')}
        columns=[x[1]for x in old_db.execute('PRAGMA table_info(facts)')]
        by_id={f['fact_id']:f for f in modified}
        for patch in patches:
            d=by_id[patch['fact_id']]['derived']
            new_db.execute('UPDATE facts SET spend_threshold_krw=?,spend_operator=?,spend_state=?,derived_json=? WHERE fact_id=?',
                (d['spend_threshold_krw'],d['spend_operator'],d['spend_state'],json.dumps(d,ensure_ascii=False,sort_keys=True),patch['fact_id']))
        new_db.execute('INSERT INTO metadata(key,value_json) VALUES (?,?)',('normalization67',json.dumps({'version':D.VERSION,'source_DB_sha256':sha(directory/'structured.sqlite'),'patched_facts':len(patches)},sort_keys=True)))
        new_db.commit()
        allowed={'spend_threshold_krw','spend_operator','spend_state','derived_json'}
        for row in new_db.execute('SELECT * FROM facts'):
            changed={columns[i] for i,(a,b)in enumerate(zip(before[row[0]],row))if a!=b}
            assert changed<=allowed
            assert bool(changed)==(row[0]in patch_by_id)
            f=by_id[row[0]]
            assert json.loads(row[columns.index('raw_json')])==f['raw_fact']
            assert json.loads(row[columns.index('derived_json')])==f['derived']
        for table in ('documents','target_aliases'):
            assert old_db.execute('SELECT * FROM '+table+' ORDER BY 1').fetchall()==new_db.execute('SELECT * FROM '+table+' ORDER BY 1').fetchall()
        assert old_db.execute("SELECT type,name,tbl_name,sql FROM sqlite_master ORDER BY name").fetchall()==new_db.execute("SELECT type,name,tbl_name,sql FROM sqlite_master ORDER BY name").fetchall()
        assert new_db.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
        assert new_db.execute('PRAGMA foreign_key_check').fetchall()==[]
    old_zero={f['fact_id']for f in facts.values()if f['derived']['spend_state']=='known'and f['derived']['spend_threshold_krw']==0}
    new_zero={f['fact_id']for f in modified if f['derived']['spend_state']=='known'and f['derived']['spend_threshold_krw']==0}
    assert old_zero==new_zero
    for original,new in zip(facts.values(),modified):
        assert {k:v for k,v in original.items()if k!='derived'}=={k:v for k,v in new.items()if k!='derived'}
        allowed_derived={'spend_threshold_krw','spend_operator','spend_state','spend_reason'}
        assert {k:v for k,v in original['derived'].items()if k not in allowed_derived}=={k:v for k,v in new['derived'].items()if k not in allowed_derived}
    for name,expected in db_phase_sources.items():assert sha(ROOT/name)==expected
    copy_manifest={'status':'prototype_copy_built_original_preserved','repair_contract_sha256':sha(BASE/'67A_DB_repair_contract.json'),
        'facts':len(modified),'changed_facts':len(patches),'changed_cards':len({p['document_id']for p in patches}),
        'raw_fact_and_relationship_fields_unchanged':True,'known_zero_fact_ids_preserved':True,'schema_unchanged':True,
        'documents_and_aliases_unchanged':True,'SQLite_JSONL_derived_consistent':True,
        'output_files_sha256':{str((prototype/name).relative_to(ROOT)):sha(prototype/name)for name in ('structured.sqlite','facts.jsonl','documents.jsonl')},
        'wall_seconds':time.perf_counter()-started,'cpu_seconds':time.process_time()-cpu,
        'process_peak_RSS_KiB':resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,'new_API_calls':0}
    save('67A_DB_copy_manifest.json',copy_manifest)
print('새 사본의 실적 파생값 보완',copy_manifest['changed_facts'],'행 /',copy_manifest['changed_cards'],'카드')

새 사본의 실적 파생값 보완 166 행 / 35 카드


## DB × 대상 매칭의 교차 재생

같은 144개 저장 계획으로 기존 DB/새 DB와 기존 매칭/대상 후보 보완을 모두 조합해 4조건을 비교한다. 새로운 LLM 추출은 아직 하지 않는다. 수치 파생값과 대상 매칭의 효과를 분리하고 기존 양성의 추가/손실 카드까지 확인한다.

In [5]:
factorial_path=BASE/'67A_DB_matcher_factorial.json'
if factorial_path.exists():
    factorial=read(factorial_path)
    assert factorial['repair_contract_sha256']==sha(BASE/'67A_DB_repair_contract.json')
else:
    f4=R.R.fact_map(prototype);all_comparisons=[];started=time.perf_counter();cpu=time.process_time()
    arms=[('DB3_baseline',directory,facts,'baseline'),('DB3_projected',directory,facts,'projected'),
          ('DB4_baseline',prototype,f4,'baseline'),('DB4_projected',prototype,f4,'projected')]
    for index,record in enumerate(plans):
        original=R.S.original_scope(record['question'],documents);outputs={};shift=index%4
        for label,db,fm,matcher in arms[shift:]+arms[:shift]:
            outputs[label]=K.route(R,record['plan'],original,db,fm,matcher=matcher,parser_failed=record['parser_status']!='ok')
        original_cards=set(old_by_query_repeat[record['query_id'],record['repeat']]['routing']['eligible_cards'])
        assert set(outputs['DB3_baseline']['eligible_cards'])==original_cards
        all_comparisons.append({'query_id':record['query_id'],'repeat':record['repeat'],'question':record['question'],
            'saved_plan':record['plan'],'original_scope':original,'outputs':outputs,
            'card_changes':{label:{'added':sorted(set(result['eligible_cards'])-original_cards),
                                  'lost':sorted(original_cards-set(result['eligible_cards']))}for label,result in outputs.items()}})
        if (index+1)%24==0:print('4条件 재생',index+1,'/144')
    factorial={'status':'replayed_awaiting_query_level_candidate_audit','records':all_comparisons,
        'repair_contract_sha256':sha(BASE/'67A_DB_repair_contract.json'),
        'wall_seconds':time.perf_counter()-started,'cpu_seconds':time.process_time()-cpu,'new_API_calls':0}
    save('67A_DB_matcher_factorial.json',factorial)
for name,expected in db_phase_sources.items():assert sha(ROOT/name)==expected
for name,expected in copy_manifest['output_files_sha256'].items():assert sha(ROOT/name)==expected
for label in ('DB3_baseline','DB3_projected','DB4_baseline','DB4_projected'):
    print(label,'양성계획',sum(bool(r['outputs'][label]['eligible_cards'])for r in factorial['records']),
          '카드목록변경',sum(bool(r['card_changes'][label]['added']or r['card_changes'][label]['lost'])for r in factorial['records']),
          '기존카드손실',sum(bool(r['card_changes'][label]['lost'])for r in factorial['records']))

4条件 재생 24 /144
4条件 재생 48 /144
4条件 재생 72 /144
4条件 재생 96 /144
4条件 재생 120 /144
4条件 재생 144 /144
DB3_baseline 양성계획 100 카드목록변경 0 기존카드손실 0
DB3_projected 양성계획 108 카드목록변경 8 기존카드손실 0
DB4_baseline 양성계획 100 카드목록변경 0 기존카드손실 0
DB4_projected 양성계획 108 카드목록변경 8 기존카드손실 0


## 67B: 같은 시점의 기존·수정 추출기 비교

DB4와 대상 후보 보완을 공통으로 고정한다. DB4는 같은 행의 명시 실적 표현을 수치로 복구한 사본이다. 67A의 카드 회수 개선은 매칭 규칙의 효과였으며 DB 수정만의 카드 회수 개선은 관찰되지 않았다.

48개 원질문 × 3회 × 기존/수정 지시문을 새 호출한다. 질문·모델·스키마·출력 한도는 같고 호출 순서를 교차한다. 수동 계획 계약은 채점 전용이며 API 입력에 포함하지 않는다. 의미상 잘못 추출해도 재시도하지 않고, 기술적 실패에만 최대 2회 물리 시도를 허용한다.

공통 helper `67_69_execution_runtime.py`는 장시간 호출의 원응답 저장·비용 상한·중단 재개를 담당한다. 노트북은 비교 행렬과 설정 선택을 담당한다. `ALLOW_NETWORK=1`일 때만 신규 API 호출이 가능하다.

In [8]:
from pathlib import Path
import importlib.util, json, os, time, resource
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
spec=importlib.util.spec_from_file_location('execution67_notebook',ROOT/'notebooks/scripts/67_69_execution_runtime.py')
T=importlib.util.module_from_spec(spec);spec.loader.exec_module(T)
BASE=ROOT/'notebooks/data/67_structured_retrieval_repair'
queries=T.read(T.R.B65/'questions.json')['queries']
assert len(queries)==48 and len({q['query_id']for q in queries})==48
baseline_budget=T.freeze_historical_budget()
contract=T.read(BASE/'67_question_plan_contract.json')
audit=T.read(BASE/'67_saved_plan_semantic_audit.json')
for obj in (contract,audit):T.E.verify_file_map(obj['source_files_sha256'],ROOT)
assert audit['question_plan_contract_raw_sha256']==T.E.file_digest(BASE/'67_question_plan_contract.json')
T.R.X.frozen(BASE/'67B_pre_API_acceptance.json',dict(status='accepted_before_new_API',
    question_contract_sha256=T.E.file_digest(BASE/'67_question_plan_contract.json'),
    saved_audit_sha256=T.E.file_digest(BASE/'67_saved_plan_semantic_audit.json'),
    selected_DB='db_v4',selected_matcher='projected',
    DB_card_recovery_gain_in_67A=0,matcher_card_recovery_plans=8,
    selection_status='development_candidate_for_extractor_comparison',manual_gold_in_provider_input=False))
sources={**T.R.M.source_hashes()}
for name in ('67_69_execution_runtime.py','67_query_repair.py','67_db_repair.py','67_test_execution_runtime.py','65_scope_repair.py'):
    path=ROOT/'notebooks/scripts'/name;sources[str(path.relative_to(ROOT))]=T.E.file_digest(path)
critical={str(p.relative_to(ROOT)):T.E.file_digest(p)for p in [
    BASE/'67B_pre_API_acceptance.json',BASE/'67_question_plan_contract.json',BASE/'67_saved_plan_semantic_audit.json',
    BASE/'shared_budget_baseline.json',ROOT/'notebooks/data/66_visible_answer_policy/completion.json',
    T.R.B65/'questions.json',*[BASE/'db_v4'/name for name in ('structured.sqlite','facts.jsonl','documents.jsonl')]]}
logical=[f"{q['query_id']}:parser:r{r}"for q in queries for r in (1,2,3)]
physical=[T.attempt_identity(i,a)for i in logical for a in (1,2)]
q_by_id={q['query_id']:q for q in queries}
for stage,variant,prompt in [(670,'old',T.OLD_PARSER_PROMPT),(671,'new',T.NEW_PARSER_PROMPT)]:
    manifest=dict(stage=stage,parser_variant=variant,model='gpt-6-luna',reasoning='medium',max_output_tokens=3000,
        source_sha256=sources,critical_data_sha256=critical,
        notebook_phase_code_sha256={'notebooks/67_structured_retrieval_repair.ipynb':dict(prefix='exp67-B-',sha256=T.phase_code_sha(ROOT/'notebooks/67_structured_retrieval_repair.ipynb','exp67-B-'))},
        allowed_identities=dict(parser=physical,answer=[]),logical_identity_by_physical={T.attempt_identity(i,a):i for i in logical for a in (1,2)},
        allowed_request_by_identity={i:T.R.P.sha([dict(role='user',content=q_by_id[i.split(':')[0]]['question'])])for i in physical},
        parser_instructions_sha256=T.R.P.sha(prompt),parser_schema_sha256=T.R.P.sha(T.R.Q.QueryPlan.model_json_schema()),
        logical_parser_requests=144,maximum_parser_requests=288,maximum_answer_requests=0,maximum_physical_requests=288,
        stage_cost_cap_usd='8',same_question_model_schema_tokens=True,fresh_contemporary_baselines=True,
        semantic_error_retry=False,manual_gold_in_provider_input=False)
    T.R.X.frozen(T.R.P.F.STAGES[stage]/'execution_manifest.json',manifest)
print('67B 준비: 288 논리 추출 / 최대576 물리, 누적 상한8달러. 신규 호출 아직0')

67B 준비: 288 논리 추출 / 최대576 물리, 누적 상한8달러. 신규 호출 아직0


In [3]:
allow_network=os.environ.get('ALLOW_NETWORK')=='1'
started=time.perf_counter();cpu=time.process_time();records=[]
for index,q in enumerate(queries):
    for repeat in (1,2,3):
        order=(670,671)if(index+repeat)%2 else(671,670)
        for stage in order:
            row=T.extract_query(stage,q,repeat,allow_network=allow_network)
            records.append(dict(parser_variant='old'if stage==670 else'new',**row))
            if len(records)%12==0:
                cost=T.budget_ledger()
                print(f"추출 {len(records)}/288, 누적 보수 차감 ${cost['budget_debit_usd']}",flush=True)
T.R.X.frozen(BASE/'67B_parser_results.json',dict(status='extracted_before_semantic_review',records=records,
    physical_order_crossed=True,manual_gold_in_provider_input=False,new_answers=0))
T.write(BASE/'67B_execution_resources.json',dict(wall_seconds=time.perf_counter()-started,cpu_seconds=time.process_time()-cpu,
    peak_rss_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,logical_results=len(records),
    ledger=T.budget_ledger(),browser_output_time='미측정',GPU='미사용',new_embeddings=0))
print('추출 완료. 의미 판정과 고정 DB/매칭 조회 비교가 다음 단계다.')

추출 12/288, 누적 보수 차감 $1.660539855
추출 24/288, 누적 보수 차감 $1.661643375
추출 36/288, 누적 보수 차감 $1.663022975
추출 48/288, 누적 보수 차감 $1.664584605
추출 60/288, 누적 보수 차감 $1.666270945
추출 72/288, 누적 보수 차감 $1.667504205
추출 84/288, 누적 보수 차감 $1.669193525
추출 96/288, 누적 보수 차감 $1.670681815
추출 108/288, 누적 보수 차감 $1.672372025
추출 120/288, 누적 보수 차감 $1.674019925
추출 132/288, 누적 보수 차감 $1.675737535
추출 144/288, 누적 보수 차감 $1.677361115
추출 156/288, 누적 보수 차감 $1.679397055
추출 168/288, 누적 보수 차감 $1.681323965
추출 180/288, 누적 보수 차감 $1.684088725
추출 192/288, 누적 보수 차감 $1.686535635
추출 204/288, 누적 보수 차감 $1.687145085
추출 216/288, 누적 보수 차감 $1.688160245
추출 228/288, 누적 보수 차감 $1.688870435
추출 240/288, 누적 보수 차감 $1.689602675
추출 252/288, 누적 보수 차감 $1.691810515
추출 264/288, 누적 보수 차감 $1.693820415
추출 276/288, 누적 보수 차감 $1.695976665
추출 288/288, 누적 보수 차감 $1.697885305
추출 완료. 의미 판정과 고정 DB/매칭 조회 비교가 다음 단계다.


## 67B 후속: 새로 추출한 계획의 고정 DB 조회

위 288개 신규 추출은 모델 응답 형식 성공이고, 의미 정답 또는 SQL 조회 성공을 뜻하지 않는다. 사전에 고정한 질문 계약으로 각 계획을 수동 검토하고, DB4·대상 후보 보완을 양쪽에 똑같이 적용한다. 같은 카드 AND와 원래 발급사·상품 범위를 유지한다. 아래의 0건도 원인별로 나누며 범용 질문의 빈 조건을 실패로 세지 않는다.

In [7]:
from collections import Counter,defaultdict
new_results=T.read(BASE/'67B_parser_results.json')['records']
assert len(new_results)==288 and len({(r['parser_variant'],r['query_id'],r['repeat'])for r in new_results})==288
assert all(r['status']=='ok'for r in new_results)
directory=BASE/'db_v4';facts=T.R.R.fact_map(directory)
documents=[json.loads(line)for line in(directory/'documents.jsonl').read_text().splitlines()]
lookup={q['query_id']:q for q in queries}
contract_files=[BASE/'67B_parser_results.json',BASE/'67B_pre_API_acceptance.json',BASE/'67_question_plan_contract.json',
    *[directory/name for name in('structured.sqlite','facts.jsonl','documents.jsonl')]]
input_contract={str(path.relative_to(ROOT)):T.E.file_digest(path)for path in contract_files}
T.R.X.frozen(BASE/'67B_lookup_contract.json',dict(files_sha256=input_contract,matcher='projected',DB='db_v4',
    notebook_code_sha256=T.phase_code_sha(ROOT/'notebooks/67_structured_retrieval_repair.ipynb','exp67-eval-'),
    same_card_AND=True,original_scope=True,manual_gold_in_query=False))
result_path=BASE/'67B_lookup_results.json'
if result_path.exists():
    lookups=T.read(result_path)['records']
else:
    lookups=[];start=time.perf_counter();cpu=time.process_time()
    for index,row in enumerate(new_results,1):
        q=lookup[row['query_id']];variant=row['parser_variant'];stage=670 if variant=='old'else 671
        rawrow=T.read(T.R.P.F.STAGES[stage]/'plans'/(row['identity']+'.json'))
        T.validate_parser_checkpoint(rawrow,q,T.read(T.R.P.F.STAGES[stage]/'execution_manifest.json'),T.R.P.F.STAGES[stage])
        assert {k:v for k,v in row.items()if k!='parser_variant'}==rawrow
        original=T.R.S.original_scope(q['question'],documents)
        routing=T.K.route(T.R,row['plan'],original,directory,facts,matcher='projected',parser_failed=row['status']!='ok')
        assert original['card_keys']is None or set(routing['eligible_cards'])<=set(original['card_keys'])
        for card,fids in routing['matching_fact_ids'].items():assert all(facts[fid]['document_id']==card for fid in fids)
        reason=('clarify_scope'if original['clarify']else'parser_technical_failure'if row['status']!='ok'else
            'unsupported_filter'if row['plan']['unsupported_filters']else'empty_groups'if not row['plan']['groups']else
            'matched'if routing['eligible_cards']else'no_matching_cards')
        lookups.append(dict(parser_variant=variant,query_id=q['query_id'],question=q['question'],repeat=row['repeat'],
            parser_row_sha256=T.R.P.sha(rawrow),original_scope=original,routing=routing,reason_category=reason))
        if index%48==0:print('고정 조회',index,'/288',flush=True)
    T.R.X.frozen(result_path,dict(records=lookups,lookup_contract_sha256=T.E.file_digest(BASE/'67B_lookup_contract.json'),
        wall_seconds=time.perf_counter()-start,cpu_seconds=time.process_time()-cpu,new_API_requests=0))
for variant in('old','new'):
    own=[r for r in lookups if r['parser_variant']==variant]
    print(variant,dict(Counter(r['reason_category']for r in own)))
print('의미 판정 전체와 개별 후보 변화 검토 후에 설정을 선택한다.')

old {'matched': 106, 'no_matching_cards': 11, 'unsupported_filter': 3, 'empty_groups': 24}
new {'matched': 120, 'no_matching_cards': 3, 'empty_groups': 21}
의미 판정 전체와 개별 후보 변화 검토 후에 설정을 선택한다.


In [7]:
from collections import Counter,defaultdict
import statistics
lookups=T.read(BASE/'67B_lookup_results.json')['records']
by_plan={(r['parser_variant'],r['query_id'],r['repeat']):r for r in lookups}
pairs=[]
for q in queries:
    for repeat in(1,2,3):
        old,new=(by_plan[variant,q['query_id'],repeat]for variant in('old','new'))
        old_cards=set(old['routing']['eligible_cards']);new_cards=set(new['routing']['eligible_cards'])
        pairs.append(dict(query_id=q['query_id'],repeat=repeat,question=q['question'],
            old_cards=sorted(old_cards),new_cards=sorted(new_cards),added=sorted(new_cards-old_cards),lost=sorted(old_cards-new_cards),
            positive_gain=bool(new_cards)and not old_cards,positive_loss=bool(old_cards)and not new_cards,
            same_card_set=old_cards==new_cards,old_reason=old['reason_category'],new_reason=new['reason_category']))
T.R.X.frozen(BASE/'67B_paired_candidate_changes.json',dict(records=pairs,
    positive_gain=sum(bool(r['positive_gain'])for r in pairs),positive_loss=sum(bool(r['positive_loss'])for r in pairs),
    same_card_sets=sum(r['same_card_set']for r in pairs),query_count=48,repeats=3,
    diagnostic_not_final_answer_accuracy=True,lookup_results_sha256=T.E.file_digest(BASE/'67B_lookup_results.json')))
resource_summary={}
for variant,stage in [('old',670),('new',671)]:
    own=[r for r in T.read(BASE/'67B_parser_results.json')['records']if r['parser_variant']==variant]
    rawfiles=sorted((T.R.P.F.STAGES[stage]/'api_cache').glob('*.json'))
    raw=[T.read(p)for p in rawfiles]
    price=T.R.P.PRICE.aggregate(T.R.P.PRICE.price_records([(str(stage),str(p.relative_to(ROOT)),T.read(p))for p in rawfiles]))
    resource_summary[variant]=dict(logical_results=len(own),physical_requests=len(raw),technical_ok=sum(r['status']=='ok'for r in own),
        parser_ready_seconds=T.R.X.timing_summary([r['parser_ready_wall_seconds']for r in own]),
        physical_request_processing_seconds=T.R.X.timing_summary([r['wall_seconds']for r in raw]),price=price,
        reason_counts=dict(Counter(r['reason_category']for r in lookups if r['parser_variant']==variant)),
        by_group={group:dict(n=len(rows:= [r for r in lookups if r['parser_variant']==variant and lookup[r['query_id']]['group']==group]),
            positive=sum(bool(r['routing']['eligible_cards'])for r in rows))for group in('explicit','general','uncertain')})
T.R.X.frozen(BASE/'67B_resources_summary.json',dict(by_variant=resource_summary,
    execution_resources_sha256=T.E.file_digest(BASE/'67B_execution_resources.json'),new_answer_requests=0,new_embeddings=0,
    browser_display_time='미측정',first_token_time='미측정'))
print('양성 전환',sum(bool(r['positive_gain'])for r in pairs),'역전',sum(bool(r['positive_loss'])for r in pairs))
print('카드 집합 동일',sum(r['same_card_set']for r in pairs),'/144')

양성 전환 14 역전 0
카드 집합 동일 125 /144


In [7]:
reviews=[];review_files=sorted((BASE/'semantic_review').glob('*.json'));source_files={}
assert len(review_files)==6
for path in review_files:
    document=T.read(path);T.E.verify_file_map(document['source_files_sha256'],ROOT)
    assert document['question_contract_sha256']==T.E.file_digest(BASE/'67_question_plan_contract.json')
    source_files[str(path.relative_to(ROOT))]=T.E.file_digest(path)
    for review in document['records']:
        source=ROOT/review['parser_source_path'];rawrow=T.read(source)
        assert T.E.file_digest(source)==review['parser_file_raw_sha256']
        assert T.R.P.sha(rawrow)==review['parser_row_sha256']and T.R.P.sha(rawrow['plan'])==review['plan_sha256']
        assert review['plan']==rawrow['plan']
        assert review['manual_entire_question_plan_and_actual_request_read']
        assert not review['sql_or_answer_results_used_for_judgment']
        reviews.append(review)
assert len(reviews)==288 and len({(r['variant'],r['query_id'],r['repeat'])for r in reviews})==288
review_map={(r['variant'],r['query_id'],r['repeat']):r for r in reviews}
summary={}
for variant in('old','new'):
    own=[r for r in reviews if r['variant']==variant]
    summary[variant]=dict(records=len(own),verdict_counts=dict(Counter(r['verdict']for r in own)),
        by_query={q['query_id']:dict(counts=dict(Counter(r['verdict']for r in own if r['query_id']==q['query_id'])),
            pass_rate=statistics.mean(r['verdict']=='PASS'for r in own if r['query_id']==q['query_id']))for q in queries},
        issue_counts=dict(Counter(issue['code']for r in own for issue in r['issues'])))
semantic_pairs=[]
for q in queries:
    old=summary['old']['by_query'][q['query_id']]['pass_rate'];new=summary['new']['by_query'][q['query_id']]['pass_rate']
    semantic_pairs.append(dict(query_id=q['query_id'],question=q['question'],old_pass_rate=old,new_pass_rate=new,
        difference=new-old,result='win'if new>old else'loss'if new<old else'tie'))
T.R.X.frozen(BASE/'67B_semantic_summary.json',dict(status='all_288_manually_reviewed_before_configuration_selection',
    by_variant=summary,paired_questions=semantic_pairs,source_files_sha256=source_files,
    manual_semantic_review=True,independent_human_gold=False,SQL_or_answer_scores_used_for_semantic_grading=False,
    no_production_promotion=True))
print({v:d['verdict_counts']for v,d in summary.items()})
print('질문별',dict(Counter(r['result']for r in semantic_pairs)))
print('후보 변화와 의미 판정은 분리하고 명시조건과 스키마 한계는 별도로 검토한다.')

{'old': {'PASS': 140, 'FAIL': 2, 'PARTIAL': 2}, 'new': {'PASS': 141, 'PARTIAL': 3}}
질문별 {'tie': 45, 'loss': 1, 'win': 2}
후보 변화와 의미 판정은 분리하고 명시조건과 스키마 한계는 별도로 검토한다.


## 선택: DB4·명시 명사 후보 매칭·기존 추출기

수정 추출기는 후보 양성14건을 늘렸지만, R09 세 번째 회차에서 기존의 명시 무료·공항 조건 보존이 소실됐다. 사전 계획의 ‘범위·조건 회귀 없이 유효 회수 개선’ 기준을 충족하지 않아 승격하지 않는다. 기존 추출기의 오류2건도 남아 있으며 기존이 더 우수하다는 결론이 아니다. DB4·좁은 명시 명사 매칭 보완은 67A의 원문/저장 계획 진단에 근거한 개발 후보로 사용한다. 운영 반영은 하지 않는다.

원래 SQL 양성 전환14를 진단 자료에 그대로 보존하되 의미 PASS인 전환13과 조건 손실을 동반한 전환1을 구분한다. 68의 답변 결과를 본 뒤 추출기를 다시 선택하지 않는다.

In [8]:
reviews=[r for path in sorted((BASE/'semantic_review').glob('*.json'))for r in T.read(path)['records']]
review_map={(r['variant'],r['query_id'],r['repeat']):r for r in reviews}
pairs=T.read(BASE/'67B_paired_candidate_changes.json')['records']
valid_gains=[r for r in pairs if r['positive_gain']and review_map['new',r['query_id'],r['repeat']]['verdict']=='PASS']
nonqualified=[r for r in pairs if r['positive_gain']and review_map['new',r['query_id'],r['repeat']]['verdict']!='PASS']
assert len(valid_gains)==13 and[(r['query_id'],r['repeat'])for r in nonqualified]==[('R09',3)]
old=review_map['old','R09',3];new=review_map['new','R09',3]
assert old['verdict']=='PASS'and new['verdict']=='PARTIAL'
assert old['plan']['unsupported_filters']and not new['plan']['unsupported_filters']
review_record=dict(status='accepted_with_new_parser_promotion_rejected',reviewer='registered Codex reviewer',
    original_major_finding='R09 repeat3 new loses preserved free airport-lounge constraint; raw positive gain is not valid recovery',
    correct_manual_source_path='notebooks/data/67_structured_retrieval_repair/semantic_review/R06_R20.json',
    original_review_source_path_typo_corrected_by_main=True,selected_parser='old',new_parser_rejected=True,
    main_agrees_with_material_finding=True,no_answer_scores_used=True,old_parser_not_claimed_superior=True,
    source_files_sha256={str(path.relative_to(ROOT)):T.E.file_digest(path)for path in[
        BASE/'67B_semantic_summary.json',BASE/'67B_lookup_results.json',BASE/'67B_paired_candidate_changes.json',
        BASE/'semantic_review/R06_R20.json']})
T.R.X.frozen(BASE/'independent_review/parser_selection_review.json',review_record)
selection_audit=dict(raw_positive_gains=14,raw_positive_losses=0,semantic_PASS_positive_gains=13,
    nonqualified_positive_gains=[dict(query_id=r['query_id'],repeat=r['repeat'],old_plan=old['plan'],new_plan=new['plan'],
        reason='explicit_free_airport_constraint_not_preserved')for r in nonqualified],
    raw_values_preserved=True,semantic_question_wins=2,semantic_question_losses=1,semantic_question_ties=45,
    selected_parser='old',reason='No-regression gate not met by new parser; retain baseline, not evidence old is superior',
    selected_DB='db_v4',matcher='projected',operating_code_changed=False)
T.R.X.frozen(BASE/'67B_selection_audit.json',selection_audit)
paths=[BASE/name for name in('67B_parser_results.json','67B_lookup_results.json','67B_lookup_contract.json',
    '67B_semantic_summary.json','67B_paired_candidate_changes.json','67B_selection_audit.json',
    '67B_pre_API_acceptance.json','67_question_plan_contract.json','67_saved_plan_semantic_audit.json',
    '67A_DB_repair_contract.json','67A_DB_matcher_factorial.json','shared_budget_baseline.json',
    'independent_review/parser_selection_review.json')]
paths.extend(sorted((BASE/'semantic_review').glob('*.json')))
paths.extend(BASE/'db_v4'/name for name in('structured.sqlite','facts.jsonl','documents.jsonl'))
for stage in(670,671):
    folder=T.R.P.F.STAGES[stage]
    paths.append(folder/'execution_manifest.json')
    paths.extend(sorted((folder/'plans').glob('*.json')))
    paths.extend(sorted((folder/'api_cache').glob('*.json')))
for name in('67_69_execution_runtime.py','67_query_repair.py','67_db_repair.py','65_scope_repair.py'):
    paths.append(ROOT/'notebooks/scripts'/name)
files={str(path.relative_to(ROOT)):T.E.file_digest(path)for path in paths}
T.R.X.frozen(BASE/'selection.json',dict(status='accepted_development_search_candidate_for68',
    parser_variant='old',matcher='projected',DB='db_v4',files_sha256=files,
    new_parser_promotion_rejected=True,scope_condition_regression='R09:r3',
    stage68_uses_no_answer_based_search_selection=True,production_promoted=False,
    remaining_limitations=['baseline parser actual errors remain','free/point/channel schema limits remain','broad candidates require original-question U and answer verification']))
print('68 선택 동결: DB4 / projected / old parser. 새 추출기 승격 거절, 운영 반영 없음.')

68 선택 동결: DB4 / projected / old parser. 새 추출기 승격 거절, 운영 반영 없음.
